# Week 2 · Notebook 2 — A model that reads history, not just today

Yesterday's MLP looked at **one day's** features and guessed tomorrow's return —
a snapshot. Today: an **LSTM**, a model that reads a whole **sequence** of days in
order, so it can pick up on patterns a snapshot can't see (a slow build-up, not
just where things stand right now).

One function you build: `to_sequences` — the reshaping step needed to train on
sequences. Then you fire the LSTM at the real backtester, same as yesterday, and
compare it head-to-head against your MLP on the identical test period.

**This notebook is also an invitation.** Once the LSTM is working, you have room —
and encouragement — to go further: more layers, a bigger hidden size, a different
architecture entirely. `models.py` is built to be extended, not just used.

## 1. Rebuild the dataset (same as yesterday)

Same feed, same features, same train/test split — no new work here, just getting
back to where you left off.

In [ ]:
import sys, os
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')
import numpy as np, json
import matplotlib.pyplot as plt
os.makedirs('dashboard/data', exist_ok=True)
from tradinglab.data_feed import DataFeed
from tradinglab.features import build_pooled_dataset

EPOCHS = 150
COMMISSION = 0.00
TOP_K = 4
SEQ_LEN = 10

feed = DataFeed.from_dir('data/egx')
# feed = DataFeed.from_dir('data/egx', symbols=['COMI', 'HRHO', 'TMGH', 'SWDY', 'FWRY', 'ABUK'])
# feed = DataFeed.from_dir('data/egx', symbols=['ABUK'])

split_day = int(feed.n_days * 0.7)
X_train, y_train, X_test, y_test = build_pooled_dataset(feed, split_day)
n_features = X_train.shape[1]
print('same dataset as yesterday:', len(X_train), 'train /', len(X_test), 'test')


## 2. Function — `to_sequences`

An MLP sees one row at a time. An LSTM needs **sequences** — overlapping windows
of `seq_len` consecutive days — so it has history to learn a pattern from, not just
a single moment.

**In:** `X` (samples, features), `y` (labels), `seq_len`.
**Out:** `X_seq` shaped `(n, seq_len, features)`, `y_seq` (label after each window).
**Hint:** slide a window of `seq_len` rows across `X`; the label is `y` at the end
of each window.
**Done when:** the shape check passes.

In [ ]:
def to_sequences(X, y, seq_len=5):
    xs, ys = [], []
    # ---8<--- solution
    for i in range(seq_len, len(X)):
        xs.append(X[i-seq_len:i]); ys.append(y[i])
    # ---8<--- end
    return np.array(xs, dtype='float32'), np.array(ys, dtype='float32')

from tradinglab.features import build_pooled_sequences

Xtr_s, ytr_s, Xte_s, yte_s = build_pooled_sequences(feed, split_day, seq_len=SEQ_LEN)
# assert Xtr_s.shape[1:] == (5, n_features), 'wrong sequence shape'
# print('to_sequences correct ✓  ->', Xtr_s.shape)


## 3. Train the LSTM
Same training loop, same overfitting check (watch train vs test loss) as yesterday
— `train_model` doesn't care whether the model is an MLP or an LSTM.

In [ ]:
import torch; torch.manual_seed(0)
from tradinglab.models import LSTMRegressor
from tradinglab.ml import train_model, predict

lstm = LSTMRegressor(n_features=n_features, hidden=32)
history = train_model(lstm, Xtr_s, ytr_s, Xte_s, yte_s, epochs=EPOCHS)

plt.figure(figsize=(9,4))
plt.plot(history['train'], label='train loss'); plt.plot(history['test'], label='test loss')
plt.yscale('log')
plt.legend(); plt.grid(alpha=.3); plt.title('LSTM — watch the gap = overfitting'); plt.show()
print('final train %.5f | test %.5f' % (history['train'][-1], history['test'][-1]))


## 4. Fire it — and notice something about the observation

Yesterday's `model_to_strategy` sliced `observation[:, -1, :]` — just today's row —
because the MLP only wants a snapshot. The LSTM wants the **whole window**.

Look at the observation's shape: `(n_assets, lookback, n_features)`. That's
*already* exactly `(batch, seq_len, features)` — precisely what an LSTM expects.
**No reshaping needed at all** — the observation you've been using since week 1
was built the right shape for this from the start.

In [ ]:
def lstm_to_strategy(model, seq_len, top_k=2):
    from tradinglab.strategies.predictor import predictions_to_weights
    def strategy(observation):
        window = observation[:, -seq_len:, :]              # take only the last seq_len days
        preds = predict(model, window.astype('float32'))
        return predictions_to_weights(preds, top_k)
    return strategy


## 5. Run it on the real backtester — and compare to yesterday's MLP

Same universe, same test period, same benchmark. The only thing different is which
model is making the calls.

In [ ]:
from tradinglab.simulator import PortfolioSimulator
from tradinglab.backtester import run_backtest
from tradinglab.report import report
from tradinglab.strategies.predictor import model_to_strategy
from tradinglab.models import MLP

sim = PortfolioSimulator(feed)
split = int(feed.n_days * 0.7)

result_lstm = run_backtest(sim, lstm_to_strategy(lstm, seq_len=SEQ_LEN, top_k=TOP_K), lookback=30, start=split)
report(result_lstm, title='LSTM strategy vs benchmark (test period)', start_capital=1000.0)

json.dump({'portfolio':[round(x,4) for x in result_lstm['portfolio'].tolist()],
           'benchmark':[round(x,4) for x in result_lstm['benchmark'].tolist()]},
          open('dashboard/data/lstm_equity.json','w'))

## 6. The honest comparison

Re-train a fresh MLP the same way as yesterday (same seed, same split) so this is
a fair, same-session comparison — then plot both against the benchmark.

In [ ]:
mlp = MLP(n_features, hidden=32)
train_model(mlp, X_train, y_train, X_test, y_test, epochs=1500)
result_mlp = run_backtest(sim, model_to_strategy(mlp, top_k=TOP_K), lookback=30, start=split)

START = 1000.0
dates = result_lstm['dates']
plt.figure(figsize=(11,5))
plt.plot(dates, result_mlp['portfolio']*START, label='MLP (yesterday)')
plt.plot(dates, result_lstm['portfolio']*START, label='LSTM (today)')
plt.plot(dates, result_lstm['benchmark']*START, label='benchmark', linestyle='--')
plt.legend(); plt.grid(alpha=.3); plt.ylabel('EGP'); plt.title('MLP vs LSTM vs benchmark')
plt.gcf().autofmt_xdate(); plt.show()

print(f"MLP final : {result_mlp['portfolio'][-1]*START:,.0f} EGP")
print(f"LSTM final: {result_lstm['portfolio'][-1]*START:,.0f} EGP")
print(f"benchmark : {result_lstm['benchmark'][-1]*START:,.0f} EGP")


## 7. Go further — this is genuinely open

More power didn't automatically mean a better result — sit with that, same as
you did yesterday. But now that you have a working LSTM, there's real room to
experiment. Ideas, roughly easiest to hardest:

- **Widen it.** Try `hidden=64` or `hidden=128` in `LSTMRegressor`.
- **Deepen it.** `nn.LSTM(..., num_layers=2)` in `models.py` — does stacking help
  or just overfit faster?
- **Change `seq_len`.** Does a longer or shorter history window change anything?
- **Try a GRU instead** (`nn.GRU` is a drop-in replacement for `nn.LSTM` in
  PyTorch) — simpler, sometimes just as good.
- **Add dropout** between the LSTM and the head to fight overfitting.

Whatever you try, the loop is always the same: change the model, retrain, check
**test** loss (not train), then fire it at the backtester and read the honest
verdict. That loop — not any particular architecture — is the actual skill.

**Graduate** `to_sequences` into `src/tradinglab/features.py`, then:
`uv run pytest week2/tests/`